# Multimodal Semiotic Relation Detection — Single-Label Dual-Encoder (`UmBERTo` + `ViT-224`) vs. Single Textual Encoder

**Task:** Single-Label 3-Class Classification (`Additiva` = 0, `Divergente` = 1, `Parallela` = 2)  
- **Text Encoder**: Domain-adapted `UmBERTo` (`./checkpoints/relation_textual_no_corr` for No Correlations; `./checkpoints/relation_textual_corr` for With Correlations).
- **Vision Encoder**: [`google/vit-base-patch16-224`](https://huggingface.co/google/vit-base-patch16-224) (768-d `[CLS]` patch representation).
- **Single-Label Classifier Head**: Multi-class `SGDClassifier(loss='log_loss', penalty='l2', random_state=42)` trained on the 1536-d joint `[CLS_text || CLS_img]` representation (Dual-Encoder) vs. the 768-d `[CLS_text]` representation (Text-Only baseline) with 1D integer targets `{0, 1, 2}`.


In [ ]:
# Install CUDA-enabled PyTorch (supports GTX 1080 Ti / Pascal via CUDA 12.1) and all requirements
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ../requirements.txt


## 1. Environment Setup & Paths


In [ ]:
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, pipeline, set_seed
import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # Force plain-text progress bars in VS Code

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

repo_root = Path.cwd().resolve()
for _ in range(5):
    if (repo_root / "datasets").exists():
        break
    repo_root = repo_root.parent
REPO_ROOT = repo_root
DATASETS_DIR = REPO_ROOT / "datasets" / "corpora"
IMAGES_DIR = DATASETS_DIR / "images"
CHECKPOINTS_DIR = Path("./checkpoints")
RESULTS_DIR = Path("./results")
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MULTIMODAL_TRAIN_CSV = DATASETS_DIR / "correctly_split" / "multimodal_train_seed42.csv"
MULTIMODAL_TEST_CSV = DATASETS_DIR / "correctly_split" / "multimodal_test_seed42.csv"

RELATION_LABELS = ["Additiva", "Divergente", "Parallela"]

LOCAL_CKPT_NO_CORR = str(CHECKPOINTS_DIR / "relation_textual_no_corr")
LOCAL_CKPT_WITH_CORR = str(CHECKPOINTS_DIR / "relation_textual_corr")

VIT_MODEL_ID = "google/vit-base-patch16-224" 


## 2. Helper Functions for Single-Label Mapping, Splitting & Feature Extraction


In [ ]:
def row_to_relation_class(row):
    """Maps row to mutually exclusive single-label class: 0=Additiva, 1=Divergente, 2=Parallela."""
    if int(row["Additiva"]) == 1:
        return 0
    elif int(row["Divergente"]) == 1:
        return 1
    elif int(row["Parallela"]) == 1:
        return 2
    return 0


def split_dataset(data: pd.DataFrame, test_size: float = 0.2, eval_size: float = 0.2, random_state: int = 42):
    """
    Identical 60/20/20 train/eval/test split used in finetune_pixtral_*.ipynb.
    Holds out the exact same 20% test_data (129 rows) as inference_pixtral.py.
    """
    train_data, test_data = train_test_split(
        data, test_size=test_size, random_state=random_state
    )
    adjusted_eval_size = eval_size / (1 - test_size)
    train_data, eval_data = train_test_split(
        train_data, test_size=adjusted_eval_size, random_state=random_state
    )
    train_data = train_data.reset_index(drop=True)
    eval_data = eval_data.reset_index(drop=True)
    test_data = test_data.reset_index(drop=True)
    print(f"Train size: {len(train_data)} | Eval (Val) size: {len(eval_data)} | Test size: {len(test_data)}")
    return train_data, test_data, eval_data


def resolve_checkpoint(local_path: Path, hf_fallback: str = "") -> str:
    """Loads the locally trained checkpoint from ./checkpoints/."""
    if not (local_path.exists() and (local_path / "config.json").exists()):
        raise FileNotFoundError(f"Local checkpoint not found at {local_path}. Please run the textual training notebook first.")
    print(f"Loading local checkpoint: {local_path}")
    return str(local_path)


def extract_text_cls_features(
    texts: list,
    model_name_or_path: str,
    device: str,
    max_length: int = 197,
    batch_size: int = 32,
) -> torch.Tensor:
    """Extracts 768-d [CLS] embeddings from the last hidden state of the text encoder."""
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    txt_model = AutoModel.from_pretrained(model_name_or_path).to(device)
    txt_model.eval()

    all_cls = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = [str(t) for t in texts[start : start + batch_size]]
            tokenized = tokenizer(
                batch_texts,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_token_type_ids=True,
                return_tensors="pt",
            ).to(device)
            outputs = txt_model(
                input_ids=tokenized.input_ids,
                attention_mask=tokenized.attention_mask,
                output_hidden_states=True,
            )
            cls_emb = outputs.hidden_states[-1][:, 0, :].cpu()
            all_cls.append(cls_emb)

    return torch.cat(all_cls, dim=0)


def load_rgb_images(img_paths: pd.Series, images_dir: Path) -> list:
    images = []
    for rel_path in img_paths:
        full_path = images_dir / str(rel_path)
        img = Image.open(full_path)
        if img.mode != "RGB":
            img = img.convert("RGB")
        images.append(img)
    return images


def extract_vit_cls_features(images: list, pipe_images, batch_size: int = 32) -> torch.Tensor:
    """Extracts 768-d [CLS] visual features from google/vit-base-patch16-224."""
    all_img_cls = []
    for start in range(0, len(images), batch_size):
        batch_imgs = images[start : start + batch_size]
        imgs_out = pipe_images(batch_imgs, return_tensors=True, framework="pt")
        batch_cls = torch.stack([t[0, 0, :].cpu() for t in imgs_out], dim=0)
        all_img_cls.append(batch_cls)
    return torch.cat(all_img_cls, dim=0)


## 3. Load Multimodal Datasets & Extract Shared `ViT-224` Image Features


In [ ]:
def build_multimodal_corr_text(df_in, emotions=False, target=False, irony=False, hate=False, relations=False):
    """
    Exact correlation prefix generator from italian_correlations.py.
    Verified 100% identical to emotions_multimodal_corr_ita.csv,
    irony_multimodal_corr_ita.csv, and relation_multimodal_corr_ita.csv.
    """
    emotion_cols = [
        "Amore", "Anticipazione", "Disgusto", "Fiducia", "Gioia",
        "Neutro", "Paura", "Rabbia", "Sorpresa", "Tristezza"
    ]
    out_texts = []
    for _, row in df_in.iterrows():
        em, tg, ih, rel = [], [], [], []
        if emotions:
            for c in emotion_cols:
                if int(row.get(c, 0)) == 1:
                    em.append(c.lower())
        if target:
            if int(row.get("Contenuto", 0)) == 1:
                tg.append("contenuto")
            if int(row.get("Contenitore", 0)) == 1:
                tg.append("contenitore")
        if irony:
            if int(row.get("Ironia", 0)) == 1:
                ih.append("ironia")
            if int(row.get("Sarcasmo", 0)) == 1:
                ih.append("sarcasmo")
        if hate:
            if int(row.get("Offensivita", 0)) == 1:
                ih.append("offensivita")
        if relations:
            if int(row.get("Additiva", 0)) == 1:
                rel.append("additiva")
            if int(row.get("Divergente", 0)) == 1:
                rel.append("divergente")
            if int(row.get("Parallela", 0)) == 1:
                rel.append("parallela")
        text = str(row["text"])
        if tg:
            text = "riguarda {} ".format(" e ".join(tg)) + text
        if ih:
            text = "contiene {} ".format(" e ".join(ih)) + text
        if rel:
            text = "ha una relazione {} ".format(" e ".join(rel)) + text
        if em:
            text = "esprime {} ".format(" e ".join(em)) + text
        text = "il seguente testo " + text
        out_texts.append(text)
    return out_texts

# Load pre-split multimodal train (516) and test (129) sets from datasets/corpora/correctly_split/
train_full_nc = pd.read_csv(MULTIMODAL_TRAIN_CSV)
test_nc = pd.read_csv(MULTIMODAL_TEST_CSV)

train_full_nc["label"] = train_full_nc.apply(row_to_relation_class, axis=1)
test_nc["label"] = test_nc.apply(row_to_relation_class, axis=1)

train_full_wc = train_full_nc.copy()
test_wc = test_nc.copy()
train_full_wc["text"] = build_multimodal_corr_text(train_full_wc, emotions=True, target=True, irony=True, hate=True, relations=False)
test_wc["text"] = build_multimodal_corr_text(test_wc, emotions=True, target=True, irony=True, hate=True, relations=False)
train_full_nc = train_full_nc.reset_index(drop=True)
test_nc = test_nc.reset_index(drop=True)
train_full_wc = train_full_wc.reset_index(drop=True)
test_wc = test_wc.reset_index(drop=True)
print(f"Multimodal Split -> Train: {len(train_full_nc)} | Test: {len(test_nc)}")

# 1D Single-Label Integer Targets (0=Additiva, 1=Divergente, 2=Parallela)
y_train = train_full_nc["label"].values.astype(int)
y_test = test_nc["label"].values.astype(int)

print(f"Loading Vision Encoder: {VIT_MODEL_ID} ...")
pipe_images = pipeline(task="image-feature-extraction", model=VIT_MODEL_ID, device=device)

print("Extracting ViT-224 features for training and test images...")
train_images = load_rgb_images(train_full_nc["img_path"], IMAGES_DIR)
test_images = load_rgb_images(test_nc["img_path"], IMAGES_DIR)

h_img_train = extract_vit_cls_features(train_images, pipe_images)
h_img_test = extract_vit_cls_features(test_images, pipe_images)
print(f"ViT Train Features shape: {h_img_train.shape} | ViT Test Features shape: {h_img_test.shape}")


## 4. Regime 1 — Without Correlations: Single Textual Encoder vs. Dual-Encoder (`UmBERTo` + `ViT-224`)


In [ ]:
ckpt_no_corr = LOCAL_CKPT_NO_CORR

h_txt_train_nc = extract_text_cls_features(train_full_nc["text"].tolist(), ckpt_no_corr, device)
h_txt_test_nc = extract_text_cls_features(test_nc["text"].tolist(), ckpt_no_corr, device)

# 4A. Blind Text-Only Baseline on Multimodal Dataset (Without Correlations) — Single-Label
clf_txt_nc = SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
clf_txt_nc.fit(h_txt_train_nc.numpy(), y_train)
y_pred_txt_nc = clf_txt_nc.predict(h_txt_test_nc.numpy()).astype(int)

print("=== [No Correlations] Blind Text-Only Baseline (129 Test Posts) ===")
print(classification_report(y_test, y_pred_txt_nc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0))

# 4B. Dual-Encoder Text + Image (Without Correlations) — Single-Label
h_joint_train_nc = torch.cat((h_txt_train_nc, h_img_train), dim=1)
h_joint_test_nc = torch.cat((h_txt_test_nc, h_img_test), dim=1)

clf_mm_nc = SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
clf_mm_nc.fit(h_joint_train_nc.numpy(), y_train)
y_pred_mm_nc = clf_mm_nc.predict(h_joint_test_nc.numpy()).astype(int)

print("=== [No Correlations] Dual-Encoder Text + Image (129 Test Posts) ===")
print(classification_report(y_test, y_pred_mm_nc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0))


## 5. Regime 2 — With Correlations: Single Textual Encoder vs. Dual-Encoder (`UmBERTo` + `ViT-224`)


In [ ]:
ckpt_with_corr = LOCAL_CKPT_WITH_CORR

h_txt_train_wc = extract_text_cls_features(train_full_wc["text"].tolist(), ckpt_with_corr, device)
h_txt_test_wc = extract_text_cls_features(test_wc["text"].tolist(), ckpt_with_corr, device)

# 5A. Blind Text-Only Baseline on Multimodal Dataset (With Correlations) — Single-Label
clf_txt_wc = SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
clf_txt_wc.fit(h_txt_train_wc.numpy(), y_train)
y_pred_txt_wc = clf_txt_wc.predict(h_txt_test_wc.numpy()).astype(int)

print("=== [With Correlations] Blind Text-Only Baseline (129 Test Posts) ===")
print(classification_report(y_test, y_pred_txt_wc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0))

# 5B. Dual-Encoder Text + Image (With Correlations) — Single-Label
h_joint_train_wc = torch.cat((h_txt_train_wc, h_img_train), dim=1)
h_joint_test_wc = torch.cat((h_txt_test_wc, h_img_test), dim=1)

clf_mm_wc = SGDClassifier(loss="squared_hinge", penalty="l2", n_jobs=-1, random_state=SEED, class_weight="balanced")
clf_mm_wc.fit(h_joint_train_wc.numpy(), y_train)
y_pred_mm_wc = clf_mm_wc.predict(h_joint_test_wc.numpy()).astype(int)

print("=== [With Correlations] Dual-Encoder Text + Image (129 Test Posts) ===")
print(classification_report(y_test, y_pred_mm_wc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0))


## 6. Final Comparison Tables & Per-Label Classification Reports


In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

os.makedirs(RESULTS_DIR, exist_ok=True)

rep_txt_nc = classification_report(y_test, y_pred_txt_nc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0)
rep_mm_nc = classification_report(y_test, y_pred_mm_nc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0)
rep_txt_wc = classification_report(y_test, y_pred_txt_wc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0)
rep_mm_wc = classification_report(y_test, y_pred_mm_wc, labels=[0, 1, 2], target_names=RELATION_LABELS, digits=4, zero_division=0)

print('=== [1/4] Text-Only (UmBERTo) — WITHOUT Correlations ===')
print(rep_txt_nc)

print('\n=== [2/4] Multimodal Dual-Encoder (UmBERTo + ViT-224) — WITHOUT Correlations ===')
print(rep_mm_nc)

print('\n=== [3/4] Text-Only (UmBERTo) — WITH Correlations ===')
print(rep_txt_wc)

print('\n=== [4/4] Multimodal Dual-Encoder (UmBERTo + ViT-224) — WITH Correlations ===')
print(rep_mm_wc)

def summarize_singlelabel(modality, name, y_true, y_pred):
    return {
        'Task': 'Semiotic Relation Classification (3-Class Single-Label)',
        'Modality': modality,
        'Notebook': 'multimodal_dual_encoder_relation.ipynb',
        'Configuration': name,
        'Micro F1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true, y_pred),
    }

results_table = pd.DataFrame([
    summarize_singlelabel('Multimodal Split (Text-Only SGD)', 'Text-Only (UmBERTo) — No Correlations', y_test, y_pred_txt_nc),
    summarize_singlelabel('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (UmBERTo + ViT-224) — No Correlations', y_test, y_pred_mm_nc),
    summarize_singlelabel('Multimodal Split (Text-Only SGD)', 'Text-Only (UmBERTo) — With Correlations', y_test, y_pred_txt_wc),
    summarize_singlelabel('Multimodal Dual-Encoder (Text + ViT-224)', 'Multimodal Dual-Encoder (UmBERTo + ViT-224) — With Correlations', y_test, y_pred_mm_wc),
])

results_table.to_csv(RESULTS_DIR / 'multimodal_dual_encoder_relation_summary.csv', index=False)
with open(RESULTS_DIR / 'multimodal_dual_encoder_relation_reports.txt', 'w', encoding='utf-8') as f_rep:
    f_rep.write('=== [1/4] Text-Only (UmBERTo) — WITHOUT Correlations ===\n' + rep_txt_nc + '\n\n')
    f_rep.write('=== [2/4] Multimodal Dual-Encoder (UmBERTo + ViT-224) — WITHOUT Correlations ===\n' + rep_mm_nc + '\n\n')
    f_rep.write('=== [3/4] Text-Only (UmBERTo) — WITH Correlations ===\n' + rep_txt_wc + '\n\n')
    f_rep.write('=== [4/4] Multimodal Dual-Encoder (UmBERTo + ViT-224) — WITH Correlations ===\n' + rep_mm_wc + '\n')

display(results_table.round(4))
